# Datamine ENGLOG Process Example

This notebook demonstrates how to configure and run the **`englog`** process wrapper in `dmstudio`.

## Process Description

## ENGLOG

The process takes as input a file containing coded drillhole logs, and translates them through a dictionary file into readable English equivalents in a report format.

It can also take input from a second file containing remarks, and join this information at the appropriate downhole distance to the output report. The report can be sent to the printer, to a system file, and to the screen.

Report contents are typically:

>SYSFILE> |  Name of system file for output report. This prompt is only given if SYSFILE=1.
---|---
>HEADING > |  Prompts for heading lines. Format of heading line input: HDn[,COL=ccc|,JL|,JC|,JR];heading text : where n = heading number (1 to 9); ccc = start column; JL,JC,JR = justify left, centre and right, respectively. : = heading text terminator if not present, last non blank character used as terminator inclusively.  Heading lines must be entered sequentially starting at 1. Partial heading lines (i.e two lines with the same heading number) should be entered from left to right. Unless overriding options are used, heading text for partial heading lines will be concatenated. Heading input must be terminated with a blank line. A blank line is required even if there are noheading lines.
>FOOTING > |  Prompts for footing lines. Format of footing line input: HDn[,COL = ccc][,JL|JC|JR];footing text : where n = footing number (1 to 9); ccc= start column; JL,JC,JR = justify left, centre and right, respectively. : = footing text terminator if not present, last non blank character used as terminator inclusively.  Footing lines must be entered sequentially starting at 1. Partial footing lines (i.e two lines with the same footing number) should be entered from left to right. Unless overriding options are used, footing text for partial footing lines will be concatenated. Footing input must be terminated with a blank line. A blank line is required even if there are no footing lines.
>FIELD> |  Name of the field to be printed. Enter blank line to terminate entry of field names.
>FORMAT> |  FORTRAN format specification for the field, including any leading or trailing blanks.
>ENGLISH_FIELD> |  Enter the name of the first English field [as defined in the IN file DD] which is to be decoded through the DICT file or transposed directly to the output report. Terminate with a blank line.
>FIELD> |  Name of the field to be printed. Enter blank line to terminate entry of field names.
>ENGLISH_TYPE> |  Enter the value of the TYPE field in the DICT file which matches the ENGLISH_FIELD. If this is left blank then the field value will be transposed directly to the output report.
>PRECEDENT> |  Enter a string of text of up to 30 characters which will precede the English field. This may be left blank.
>ANTECEDENT> |  Enter a string of text of up to 30 characters which will follow the English field. This may be left blank.
>FORMAT> |  If the English field is to be transposed rather than decoded then the format for the output report must be defined. This should be an A format for alpha fields or F or I for numerics.

### Input Files:

* **in** (Undefined):
  The input data file containing the coded log. This must contain at least the BHID and
  FROM fields. All fields which are to be decoded [the English fields] must be of the same
  type [alpha or numeric] and the same length [if alpha].
  Required=Yes

* **dict** (Undefined):
  The dictionary file containing the translated codes. It must contain the 3 fields TYPE,
  CODE and TEXT.
  Required=No

* **remarks** (Undefined):
  The remarks file contains the three fields BHID, FROM and TEXT, and should be sorted on
  BHID and FROM. The TEXT field is multi- character alpha.
  Required=No

### Output Files:

### Fields:

### Parameters:

* **lhmargin**:
  Start column for printing (1).
  Range=1,79
  Values=Undefined
  Default=1
  Required=No

* **rhmargin**:
  End column for printing (79).
  Range=1,79
  Values=Undefined
  Default=79
  Required=No

* **lines**:
  Number of lines per page of output (0). 0 - no paging.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **noff**:

* **Options** ((0): Show form feeds.; 1: Suppresses form feeds.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **double**:

* **Options** ((0): Single spacing [default];; 1: Double spacing.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **sysfile**:

* **Options** ((0): Send report to print file.; 1: Send report to a system file rather than):
  the print file. The file name is requested interactively.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **eng_marg**:
  The number of spaces left as a margin on the lefthand side of the output report before
  the decoded text is written. Default is (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **eng_leng**:
  The number of characters per line for the translated text part of the output report.
  This does not include the spaces defined by ENG_MARG.(79)
  Range=1,79
  Values=Undefined
  Default=79
  Required=No

* **precdent**:
  This parameter controls the output of the precedent. The precedent itself is defined
  interactively. Note that this parameter affects the printing of antecedents in the same
  way. Options: 0: If there is no code in the IN file [ie if it is blank for an alpha
  field or '-' for numeric] then the precedent is not included in the report. (0); 1: The
  precedent [if it has been defined] will always appear in the output report, even if the
  coded field to which it applies is absent data.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **nocomma**:
  This parameter controls the output of a comma following each ENGLISH_FIELD . Options:

* **0** (a comma will be printed after each field, unless an antecedent has been specified):
  (0); 1: there will be no automatic printing of commas. If required, they must be
  specified as antecedents.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **nonl**:
  This parameter controls the output of a new-line following each drillhole interval.

* **Options** (0: a new-line will be output after each interval. (0); 1: a new-line will not):
  be output after each interval.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **showcode**:
  This parameter controls the output of codes if no english translation is given.

* **Options** (0: the code is ignored, i.e. treated as absent. (0); 1: the code is printed):
  without translation
  Range=0,1
  Values=0,1
  Default=0
  Required=Yes

* **print**:
  Display control: (0) =-1 : Suppress output. =0 : Display output.
  Range=-1,0
  Values=-1,0
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('englog')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute englog
print("Running englog...")
dm_cmd.englog(
    in_i='t_assays',  # required
    showcode_p='required_val',  # required
    # dict_i='optional',  # optional
    # remarks_i='optional',  # optional
    # lhmargin_p=1,  # optional
    # rhmargin_p=79,  # optional
    # lines_p=0,  # optional
    # noff_p=0,  # optional
    # double_p=0,  # optional
    # sysfile_p=0,  # optional
    # eng_marg_p=0,  # optional
    # eng_leng_p=79,  # optional
    # precdent_p=0,  # optional
    # nocomma_p=0,  # optional
    # nonl_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("englog execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_englog_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")